# 04 — Keyword Density

Visualises the distribution of the **top 20 diagnostic keywords** across the five LDA topics identified in `03_topic_summary.ipynb`. Each keyword's probability weight across topics comes from pyLDAvis's `token.table` (p(topic | term) normalised to sum ≈ 1).

The 20 terms are chosen as:
- **4 exclusive signature terms per topic** (highest loglift, nearly all weight in one topic)
- **4 bridge terms** shared across 2-3 topics (showed cross-topic conceptual overlap)

## 1. Imports & Font

In [ ]:
import json, re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec

_cjk = ['PingFang SC', 'Heiti SC', 'STHeiti', 'SimHei', 'Noto Sans CJK SC']
_avail = {f.name for f in fm.fontManager.ttflist}
_font  = next((f for f in _cjk if f in _avail), None)
if _font:
    matplotlib.rcParams['font.family'] = _font
    print(f'Using CJK font: {_font}')
else:
    print('No CJK font — Chinese labels may render as boxes')
matplotlib.rcParams['axes.unicode_minus'] = False


## 2. Load LDA Data from `lda_vis_chn.html`

In [ ]:
with open("../output/lda_vis_chn.html") as f:
    html = f.read()

match    = re.search(r'var ldavis_\w+_data = ({.*?});', html, re.DOTALL)
lda_data = json.loads(match.group(1))
tt       = lda_data["token.table"]

# p(topic | term) matrix from token.table
df_tt = pd.DataFrame({"term": tt["Term"], "topic": tt["Topic"], "prob": tt["Freq"]})
pivot = df_tt.pivot_table(index="term", columns="topic",
                           values="prob", aggfunc="sum", fill_value=0)
pivot.columns = [f"T{c}" for c in pivot.columns]
print(f"Token-table: {len(pivot)} unique terms across 5 topics")


## 3. Select Top 20 Keywords & Build Weight Matrix

In [ ]:
# 4 exclusive signature terms per topic (high loglift, nearly all mass in 1 topic)
exclusive = {
    "T1": ["出借",  "补救措施", "公司财务", "授权书"],
    "T2": ["侵权行为","肖像权",  "致歉",    "二维码"],
    "T3": ["在编",  "工资标准", "体校",    "离队"],
    "T4": ["公派",  "发号",    "主体资格", "退役费"],
    "T5": ["护理费","椎间盘",  "营养费",  "伤残"],
}
# 4 bridge terms: distributed across 2-3 topics, show conceptual linkage
bridge = ["择业", "补偿金", "补发", "行政诉讼法"]

ordered = (exclusive["T1"] + exclusive["T2"] + exclusive["T3"] +
           exclusive["T4"] + exclusive["T5"] + bridge)

matrix = pivot.reindex(ordered, fill_value=0)
print("Keyword × topic weight matrix (p(topic|term)):")
print(matrix.round(3).to_string())


## 4. Topic Colour Map & Labels

In [ ]:
COLORS = {
    "T1": "#8E44AD",  # purple  — Commercial Contract & Debt
    "T2": "#C0392B",  # red     — Online Infringement / Image Rights
    "T3": "#1A9E6A",  # green   — Sports Personnel & Wages
    "T4": "#E67E22",  # orange  — Retirement & Administrative Litigation
    "T5": "#2980B9",  # blue    — Sports Injury & Medical Compensation
}

TOPIC_LABELS = {
    "T1": "T1\n商业合同违约\n与债务追偿",
    "T2": "T2\n网络侵权\n肖像权纠纷",
    "T3": "T3\n运动队人事\n薪资待遇",
    "T4": "T4\n退役安置\n行政诉讼",
    "T5": "T5\n运动伤害\n医疗赔偿",
}

# Row group labels (which topic block each term belongs to)
ROW_GROUP = {t: grp for grp, terms in exclusive.items() for t in terms}
for t in bridge:
    ROW_GROUP[t] = "bridge"


## 5. Keyword Density Heatmap

Each cell shows p(topic | keyword). Rows are ordered by topic group. Exclusive terms have a single bright cell; bridge terms show distributed colour across columns. A right-side bar encodes the maximum weight for each term.

In [ ]:
n_terms  = len(ordered)
n_topics = 5
topics   = ["T1", "T2", "T3", "T4", "T5"]

fig = plt.figure(figsize=(13, 11))
fig.patch.set_facecolor("#f8f9fa")
gs  = GridSpec(1, 2, width_ratios=[14, 1], wspace=0.03, figure=fig)
ax  = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1])

ax.set_facecolor("#f8f9fa")
ax_bar.set_facecolor("#f8f9fa")

Z = matrix[topics].values  # shape (20, 5)

# Draw each cell individually with the column's topic colour
for col_idx, tc in enumerate(topics):
    base = mcolors.to_rgb(COLORS[tc])
    for row_idx in range(n_terms):
        val  = Z[row_idx, col_idx]
        # Interpolate: white (0) -> topic colour (1)
        cell_color = tuple(1 - val * (1 - c) for c in base)
        rect = plt.Rectangle([col_idx - 0.5, row_idx - 0.5], 1, 1,
                              facecolor=cell_color, edgecolor="white", linewidth=1.2)
        ax.add_patch(rect)
        # Annotate non-zero cells
        if val > 0.05:
            txt_color = "white" if val > 0.55 else "#2C3E50"
            ax.text(col_idx, row_idx, f"{val:.2f}",
                    ha="center", va="center", fontsize=7.5,
                    color=txt_color, fontweight="bold")

# Horizontal separators between topic groups (after every 4 rows, and before bridge)
for sep in [3.5, 7.5, 11.5, 15.5, 19.5]:
    lw = 2.5 if sep == 19.5 else 1.8
    ls = "--" if sep == 19.5 else "-"
    ax.axhline(sep, color="#555", linewidth=lw, linestyle=ls, alpha=0.6)

# Y-axis: term labels, coloured by owning topic
ax.set_ylim(-0.5, n_terms - 0.5)
ax.set_yticks(range(n_terms))
ax.set_yticklabels(ordered, fontsize=10)
for tick, term in zip(ax.get_yticklabels(), ordered):
    grp = ROW_GROUP.get(term, "bridge")
    tick.set_color(COLORS[grp] if grp in COLORS else "#555555")
    tick.set_fontweight("bold" if grp != "bridge" else "normal")
ax.invert_yaxis()

# X-axis: topic column headers
ax.set_xlim(-0.5, n_topics - 0.5)
ax.set_xticks(range(n_topics))
ax.set_xticklabels([TOPIC_LABELS[t] for t in topics],
                    fontsize=8.5, linespacing=1.4)
for tick, tc in zip(ax.get_xticklabels(), topics):
    tick.set_color(COLORS[tc])
    tick.set_fontweight("bold")
ax.xaxis.set_tick_params(length=0)
ax.xaxis.tick_top()
ax.tick_params(axis="y", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)

# Right-side marginal bar: max weight per term, coloured by owning topic
max_weights = Z.max(axis=1)
bar_colors  = [COLORS.get(ROW_GROUP.get(t, "bridge"), "#888888") for t in ordered]
ax_bar.barh(range(n_terms), max_weights, color=bar_colors, alpha=0.75,
            edgecolor="white", linewidth=0.8, height=0.65)
ax_bar.set_xlim(0, 1.05)
ax_bar.set_ylim(-0.5, n_terms - 0.5)
ax_bar.invert_yaxis()
ax_bar.set_xticks([0, 0.5, 1.0])
ax_bar.set_xticklabels(["0", ".5", "1"], fontsize=7, color="#666")
ax_bar.set_yticks([])
ax_bar.set_xlabel("Max\nweight", fontsize=7, color="#666", labelpad=2)
ax_bar.tick_params(axis="x", length=2, colors="#999")
for spine in ["top", "right", "left"]:
    ax_bar.spines[spine].set_visible(False)
ax_bar.spines["bottom"].set_color("#ccc")

# Horizontal separators on bar chart too
for sep in [3.5, 7.5, 11.5, 15.5, 19.5]:
    ax_bar.axhline(sep, color="#555", linewidth=1.2, linestyle="-", alpha=0.4)

# Left-side group bracket labels
group_mids = {"T1": 1.5, "T2": 5.5, "T3": 9.5, "T4": 13.5, "T5": 17.5, "bridge": 21.5}
group_names = {
    "T1": "商业合同", "T2": "侵权肖像",
    "T3": "人事薪资", "T4": "退役行政",
    "T5": "医疗赔偿", "bridge": "跨域词"
}
for grp, mid in [("T1",1.5),("T2",5.5),("T3",9.5),("T4",13.5),("T5",17.5)]:
    ax.text(-0.85, mid, group_names[grp], ha="center", va="center",
            fontsize=8, color=COLORS[grp], fontweight="bold", rotation=90)
ax.text(-0.85, 21.5, "Bridge", ha="center", va="center",
        fontsize=7.5, color="#555", fontweight="bold", rotation=90)

# Legend
handles = [mpatches.Patch(color=COLORS[t],
            label=f"{TOPIC_LABELS[t].replace(chr(10)," "]} ({[23.9,23.9,22.8,19.7,9.7][i]:.1f}%)")
           for i, t in enumerate(topics)]
handles.append(mpatches.Patch(color="#888888", label="Bridge terms (cross-topic)"))
ax.legend(handles=handles, loc="lower center", ncol=3, fontsize=7.5,
          framealpha=0.9, edgecolor="#ccc",
          bbox_to_anchor=(0.5, -0.08))

fig.suptitle(
    "Keyword Density across LDA Topics — Athlete Court Cases\n"
    "Cell colour intensity = p(topic | keyword) from pyLDAvis token.table",
    fontsize=11, fontweight="bold", y=1.01
)
plt.savefig("../output/keyword_density_heatmap.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved → ../output/keyword_density_heatmap.png")


## 6. Stacked Distribution Bars

Same data as an **inverted stacked bar chart**: each horizontal bar is one keyword, and the coloured segments show the fraction of its probability mass assigned to each topic. Bridge terms (bottom section) visibly span multiple colour segments.

In [ ]:
fig2, ax2 = plt.subplots(figsize=(11, 9))
fig2.patch.set_facecolor("#f8f9fa")
ax2.set_facecolor("#f8f9fa")

# Row-normalise so bars always sum to 1 (stacked proportion)
row_sums = Z.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
Z_norm = Z / row_sums

y_pos = np.arange(n_terms)
lefts = np.zeros(n_terms)

for col_idx, tc in enumerate(topics):
    widths = Z_norm[:, col_idx]
    bars = ax2.barh(y_pos, widths, left=lefts,
                    color=COLORS[tc], edgecolor="white", linewidth=0.8,
                    height=0.65, label=TOPIC_LABELS[tc].replace("\n", " "))
    # Label segments > 12%
    for i, (w, l) in enumerate(zip(widths, lefts)):
        if w > 0.12:
            ax2.text(l + w / 2, i, f"{w:.0%}",
                     ha="center", va="center", fontsize=7,
                     color="white", fontweight="bold")
    lefts = lefts + widths

# Separator lines between topic groups
for sep in [3.5, 7.5, 11.5, 15.5, 19.5]:
    lw = 2.2 if sep == 19.5 else 1.4
    ls = "--" if sep == 19.5 else "-"
    ax2.axhline(sep, color="#555", linewidth=lw, linestyle=ls, alpha=0.55)

# Y-axis labels
ax2.set_yticks(y_pos)
ax2.set_yticklabels(ordered, fontsize=10)
for tick, term in zip(ax2.get_yticklabels(), ordered):
    grp = ROW_GROUP.get(term, "bridge")
    tick.set_color(COLORS[grp] if grp in COLORS else "#555555")
    tick.set_fontweight("bold" if grp != "bridge" else "normal")
ax2.invert_yaxis()

ax2.set_xlim(0, 1.0)
ax2.set_xlabel("Proportion of keyword weight assigned to each topic", fontsize=9)
ax2.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax2.tick_params(axis="x", labelsize=8)
ax2.tick_params(axis="y", length=0)
for spine in ["top", "right"]:
    ax2.spines[spine].set_visible(False)
ax2.spines["left"].set_color("#ccc")
ax2.spines["bottom"].set_color("#ccc")

# Group bracket labels on left
for grp, mid in [("T1",1.5),("T2",5.5),("T3",9.5),("T4",13.5),("T5",17.5)]:
    ax2.text(-0.08, mid, group_names[grp], ha="center", va="center",
             fontsize=8, color=COLORS[grp], fontweight="bold", rotation=90,
             transform=ax2.get_yaxis_transform())
ax2.text(-0.08, 21.5, "Bridge", ha="center", va="center",
         fontsize=7.5, color="#555", fontweight="bold", rotation=90,
         transform=ax2.get_yaxis_transform())

handles2 = [mpatches.Patch(color=COLORS[t],
             label=TOPIC_LABELS[t].replace("\n", " ")) for t in topics]
ax2.legend(handles=handles2, loc="lower right", fontsize=8,
           framealpha=0.9, edgecolor="#ccc", ncol=2)

ax2.set_title(
    "Keyword Topic Affinity — Stacked Distribution\n"
    "Each bar = one keyword; segments show proportion of weight per topic",
    fontsize=11, fontweight="bold", pad=10
)
plt.tight_layout()
plt.savefig("../output/keyword_density_bars.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved → ../output/keyword_density_bars.png")
